# Gamil RAG project

### Step 1: Import library

In [1]:
import os
import ollama
from openai import OpenAI
import json
from dotenv import load_dotenv
from huggingface_hub import login
from sentence_transformers import SentenceTransformer
import gradio as gr
from gmail.gmail_function import login_gmail, get_emails_list
import tqdm
from IPython.display import display, Markdown

/home/junmisoh/projects/gmail_rag_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: setting variables and environment

In [5]:
load_dotenv(override = True)
ollama_api_key = os.getenv("OLLAMA_API_KEY")
hf_token = os.getenv("HF_TOKEN")
credential_path = r'.credentials/credentials.json' # create credentials from Gmail API
token_path = r'.credentials/token.json' # Token will be created the first time of login to be used for subsequent login
login(hf_token, add_to_git_credential=True)

# setup ollama
ollama = OpenAI(base_url = "http://localhost:11434/v1/", api_key = ollama_api_key)
llm_model = "gemma3:4b"

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


### Step 3: Check if gmail credentials exist

In [7]:
if not os.path.exists(credential_path):
    print(f"Error: Credentials file '{credential_path}' does not exist.")
else:
    print("Credential file exists!")

Credential file exists!


### Step 4: Login gmail with credentials or token

In [8]:
service = login_gmail(credential_path, token_path)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=343023705168-4cfugldc10jktt3p54uetc55js8qha5s.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A32881%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=3UMYNLVwzAJQsLD6uxIZ2XIf5kr8wx&code_challenge=znHS6sGOxUgnaZAkz8OL-zzev1cFiVAJE2MzSAOkoEY&code_challenge_method=S256&access_type=offline


gio: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=343023705168-4cfugldc10jktt3p54uetc55js8qha5s.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A32881%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=3UMYNLVwzAJQsLD6uxIZ2XIf5kr8wx&code_challenge=znHS6sGOxUgnaZAkz8OL-zzev1cFiVAJE2MzSAOkoEY&code_challenge_method=S256&access_type=offline: Operation not supported


### Step 5: Retrieve Gmail for the past 7 days

In [9]:
emails = get_emails_list(service, days=7)

In [10]:
print(emails)

[{'title': 'Security alert', 'sender': 'Google <no-reply@accounts.google.com>', 'date_received': '2026-03-19 10:44:46', 'body': "[image: Google]\r\nYou allowed gmail_rag_project access to some of your Google Account data\r\n\r\n\r\njunming1202@gmail.com\r\n\r\nIf you didn't allow gmail_rag_project access to some of your Google Account\r\ndata, someone else may be trying to access your Google Account data.\r\n\r\nTake a moment now to check your account activity and secure your account.\r\nCheck activity\r\n<https://accounts.google.com/AccountChooser?Email=junming1202@gmail.com&continue=https://myaccount.google.com/alert/nt/1773888286000?rfn%3D127%26rfnc%3D1%26eid%3D59728158742604356%26et%3D0>\r\nTo make changes at any time to the access that gmail_rag_project has to\r\nyour data, go to your Google Account\r\n<https://accounts.google.com/AccountChooser?Email=junming1202@gmail.com&continue=https://myaccount.google.com/connections/overview/AS962fsJSkHbmSCS-JfL8rg-9xlAALPnpmI8vfMAvfQ-tqnUP7

### Step 6: Summarize gmail in the past 7 days 

In [11]:
system_prompt = f"""
You are a helpful chatbot that reads gmail content for 7 days that has been preprocessed and filtered below. 
Help user to answer any question related to their gmail based on the context. If you dont know, say so.
Here are the gmail content:

f{emails}
"""

user_prompt = """
Please give me a high-level summary of my gmail in bullet point form with appropriate MarkDown.
The bullet points should start from the most important info to the less important info.
"""

messages = [{"role":"system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]

In [12]:
response = ollama.chat.completions.create(model = llm_model, messages = messages)



In [13]:
display(Markdown(response.choices[0].message.content))

Okay, here’s a high-level summary of your Gmail inbox, presented in bullet points with Markdown formatting, prioritizing the most significant information first.

**Overall Summary:**

*   **High Volume of Notifications & Alerts:** You’re receiving a significant number of emails across various services and platforms, predominantly from:
    *   **GitHub:** Frequent alerts regarding device verification, sudo codes, and welcome messages. (4 messages)
    *   **LM Studio:**  Updates, announcements about LM Link, and a welcome email. (2 messages)
    *   **Interactive Brokers:** Tax form notifications and reporting alerts. (1 message)
    *   **DBS PayLah!:** Transaction alerts. (1 message)
    *   **Foundit:** Job opportunity notifications. (2 messages)
    *   **General Email Providers:** Multiple emails from companies like IBM and other generic services. (1 message)

*   **Security-Related Alerts:** You’ve been subject to device verification prompts from GitHub at least twice, highlighting the need to maintain secure access to your accounts.

*   **Service Updates & Onboarding:** You’ve recently been onboarded with LM Studio and possibly other services through Foundit.

**Details & Breakdown:**

*   **GitHub Activity:** The most frequent sender, focusing on device security and authentication challenges.  You've received:
    *   Sudo codes for access.
    *   Verification prompts to confirm your device.
    *   Welcome messages and updates on the platform.

*   **LM Studio Engagement:**  You’re exploring and utilizing the LM Studio platform, receiving updates about its features (LM Link) and onboarding communications.

*   **Financial Alerts:** You are actively monitored for financial transactions with Interactive Brokers, including tax form updates.

*   **Job Opportunity Notifications:** You are receiving information regarding potential employment opportunities via Foundit.

*   **Transactional Alerts:** You've enabled alerts for your DBS PayLah! account.

**Note:** *This summary is based solely on the provided email list.  A full analysis would require access to the actual email content and frequency*

To help me refine this summary further, could you tell me:

*   What is the context of this Gmail inbox? (e.g., a personal account, a professional account, a specific project?)
*   Are there any particular themes or types of emails you’d like me to emphasize?

### Step 8: Create a chatbot for user to ask further questions

In [14]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] +\
                [{"role":h["role"], "content": h["content"]} for h in history]+\
                    [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model = llm_model, messages = messages)
    return response.choices[0].message.content

In [15]:
with gr.Blocks() as ui:
    with gr.Row():
        gr.ChatInterface(fn=chat)
        
ui.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [41]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\queueing.py", line 766, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\route_utils.py", line 355, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\blocks.py", line 2158, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\blocks.py", line 1632, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\JM\projects\gmail_rag_project\.venv\Lib\site-packages\gradio\utils.py", line 1007, in async_wrapper
    response = await f(*args, **k